We are going to compute the eigenvalues and eigenvectors of the dataset.

In [ ]:
from pathlib import Path
from graph2mat4abn.tools.tbplas_tools import add_orbitals, add_hopping_terms, extract_onsites_from_coo, extract_hoppings_from_coo, compute_k_len, select_kpath
from scipy.sparse import save_npz
from tqdm import tqdm

import sisl
import tbplas as tb
import numpy as np

In [ ]:
# Create directory

eigen_dir = Path("../dataset_eigen") # Assuming this notebook is in ./graph2mat4abn/notebooks
eigen_dir.mkdir(exist_ok=True)

# Load the paths to all structures
dataset_dir = Path("../dataset")
use_only_x_atoms = ["2", "8"]
subsets = [subset for subset in dataset_dir.glob("*/") if subset.parts[-1].split("_")[-2] in use_only_x_atoms]
subsets

In [ ]:
# Iterate for each subset of SHARE_OUTPUTS_X_ATOMS, then through each structure to finally save each diagonalization.
for subset in tqdm(subsets):
    for structure_path in tqdm(subset.glob("*/")):

        file = sisl.get_sile(structure_path / "aiida.fdf")
        geometry = file.read_geometry()
        positions = geometry.xyz # Angstroms
        labels = [[orb.name() for orb in atom] for atom in geometry.atoms]
        hamiltonian_coo = file.read_hamiltonian().tocsr().tocoo()

        n_atoms = int(structure_path.parts[-2].split("_")[-2])
        vectors = geometry.cell
        cell = tb.PrimitiveCell(vectors, unit=tb.ANG)


        # Construct tbplas cell
        onsites = extract_onsites_from_coo(hamiltonian_coo)
        add_orbitals(cell, positions, onsites, labels)

        iscs, orbs_in, orbs_out, hoppings = extract_hoppings_from_coo(hamiltonian_coo, n_atoms, geometry)
        add_hopping_terms(cell, iscs, orbs_in, orbs_out, hoppings)

        # Construct tbplas overlap
        overlap_coo = file.read_overlap().tocsr().tocoo()
        overlap_cell = tb.PrimitiveCell(cell.lat_vec, cell.origin, 1.0)

        onsites = extract_onsites_from_coo(overlap_coo)
        add_orbitals(overlap_cell, positions, onsites, labels)

        iscs, orbs_in, orbs_out, hoppings = extract_hoppings_from_coo(overlap_coo, n_atoms, geometry)
        add_hopping_terms(overlap_cell, iscs, orbs_in, orbs_out, hoppings)


        # Compute the paths
        k_path, k_idx, k_label = select_kpath(n_atoms, cell, n_kpoints=20, structure=None)
        

        # Compute the bands
        solver = tb.DiagSolver(cell, overlap_cell)
        solver.config.k_points = k_path
        bands, states = solver.calc_states()
        k_len = compute_k_len(k_path)
        solver.config.prefix = "bands"
        
        # Save the results
        save_dir = eigen_dir / structure_path.parts[-2] / structure_path.parts[-1]
        save_dir.mkdir(parents=True, exist_ok=True)
        np.savez(save_dir/"bands.npz", path=str(structure_path), bands=bands)
        np.savez(save_dir/"k_path.npz", path=str(structure_path), k_path=k_path, k_idx=k_idx, k_label=k_label, k_len=k_len)
        np.savez(save_dir/"states.npz", path=str(structure_path), states=states)
        save_npz(save_dir/"overlap.npz", overlap_coo)
        
